# Cleaning the CO₂ dataset

This notebook applies the decisions documented during data inspection.

The cleaning process will:
1. Preserve the original dataset and create a working copy.
2. Apply the geographic screening rule.
3. Restrict observations to 1990–2022.
4. Select the identifiers, target and three predictors.
5. Exclude observations missing any required modelling variable.
6. Verify and save the resulting dataset.

The expected result is 5,358 observations across 164 countries and
territories.

Complete-case selection avoids filling entire missing predictor
histories, but may introduce selection bias. The excluded observations
will be recorded, and results will not be assumed to represent excluded
entities (Sterne et al., 2009).

No automatic outlier removal, scaling or statistical imputation will
be performed at this stage.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

file_path = Path("../data/raw/owid_co2.csv")

if not file_path.exists():
    file_path = Path("data/raw/owid_co2.csv")

df = pd.read_csv(file_path)

clean_df = df.copy()

print("Original shape:", df.shape)
print("Working-copy shape:", clean_df.shape)

Original shape: (56269, 79)
Working-copy shape: (56269, 79)


## Applying the geographic scope and study period

The geographic screening rule retains entities whose supplied code
contains three uppercase letters. This is an operational scope covering
countries and territories; it is not a sovereignty classification.

Entities without qualifying codes, including Kosovo, are outside this
scope. Our World in Data also provides regional and world aggregates, which are not
appropriate alongside individual entities in this analysis


The period is restricted to 1990–2022. Inspection showed that later
starting years provide little improvement in completeness while losing
many observations. GDP availability determines the 2022 endpoint.

Row counts are recorded after each selection step.

In [2]:
cleaning_log = [{
    "stage": "Original dataset",
    "rows": len(clean_df),
    "entities": clean_df["country"].nunique()
}]

geographic_mask = (
    clean_df["iso_code"]
    .astype("string")
    .str.fullmatch(r"[A-Z]{3}", na=False)
)

clean_df = clean_df.loc[geographic_mask].copy()

cleaning_log.append({
    "stage": "After geographic screening",
    "rows": len(clean_df),
    "entities": clean_df["country"].nunique()
})

print("Rows:", len(clean_df))
print("Entities:", clean_df["country"].nunique())

Rows: 48306
Entities: 223


In [3]:
clean_df = clean_df.loc[
    clean_df["year"].between(1990, 2022)
].copy()

cleaning_log.append({
    "stage": "After selecting 1990–2022",
    "rows": len(clean_df),
    "entities": clean_df["country"].nunique()
})

print("Rows:", len(clean_df))
print("Entities:", clean_df["country"].nunique())
print("Year range:", clean_df["year"].min(), "to", clean_df["year"].max())

Rows: 7359
Entities: 223
Year range: 1990 to 2022


## Selecting identifiers and modelling variables

The identifiers country, year and iso_code are retained for checking,
grouping and evaluation design. Their inclusion in this table does not
mean they will automatically be used as predictors.

The target is co2. The initial predictors are total_energy_supply,
population and gdp.

Other emissions measures, components and emissions-derived ratios are
excluded from the initial modelling table to avoid giving the model
information derived from the outcome it is intended to estimate
(Kaufman, Rosset and Perlich, 2011).

The existing numerical types and documented units are retained.

In [4]:
identifier_columns = ["country", "year", "iso_code"]

target_column = "co2"

predictor_columns = [
    "total_energy_supply",
    "population",
    "gdp"
]

required_columns = [target_column] + predictor_columns

selected_columns = identifier_columns + required_columns

clean_df = clean_df[selected_columns].copy()

print("Selected columns:")
print(clean_df.columns.tolist())

display(clean_df.head())

Selected columns:
['country', 'year', 'iso_code', 'co2', 'total_energy_supply', 'population', 'gdp']


,country,year,iso_code,co2,total_energy_supply,population,gdp
240,Afghanistan,1990,AFG,2.038982,30.196873,12045664.0,1.306598e+10
241,Afghanistan,1991,AFG,1.932622,12.496816,12238879.0,1.204736e+10
242,Afghanistan,1992,AFG,1.500374,8.207146,13278983.0,1.267754e+10
243,Afghanistan,1993,AFG,1.505263,8.155350,14943175.0,9.834582e+09
244,Afghanistan,1994,AFG,1.468623,7.871012,16250800.0,7.919856e+09


## Applying the complete-case rule

Rows missing the target or any of the three required predictors will
be excluded from the initial modelling dataset.

An exclusion table will retain these observations and identify their
missing variables. Missing values will not be replaced with zero,
an overall average or interpolated values.

Complete observations from partially covered entities will remain.
The resulting dataset will therefore be an unbalanced panel.

This is a documented sample-selection decision, not evidence that
missingness is random or that excluded entities are unimportant
(Sterne et al., 2009).

In [5]:
missing_required = clean_df[required_columns].isna()

complete_mask = ~missing_required.any(axis=1)

excluded_rows = clean_df.loc[~complete_mask].copy()

excluded_rows["missing_variables"] = (
    missing_required.loc[~complete_mask]
    .apply(
        lambda row: ", ".join(row.index[row]),
        axis=1
    )
)

rows_before = len(clean_df)

clean_df = clean_df.loc[complete_mask].copy()

cleaning_log.append({
    "stage": "After complete-case selection",
    "rows": len(clean_df),
    "entities": clean_df["country"].nunique()
})

print("Rows before:", rows_before)
print("Rows excluded:", len(excluded_rows))
print("Rows retained:", len(clean_df))
print("Entities retained:", clean_df["country"].nunique())

display(
    excluded_rows[
        ["country", "year", "missing_variables"]
    ].head(10)
)

Rows before: 7359
Rows excluded: 2001
Rows retained: 5358
Entities retained: 164


,country,year,missing_variables
1420,Andorra,1990,"total_energy_supply, gdp"
1421,Andorra,1991,"total_energy_supply, gdp"
1422,Andorra,1992,"total_energy_supply, gdp"
1423,Andorra,1993,"total_energy_supply, gdp"
1424,Andorra,1994,"total_energy_supply, gdp"
1425,Andorra,1995,"total_energy_supply, gdp"
1426,Andorra,1996,"total_energy_supply, gdp"
1427,Andorra,1997,"total_energy_supply, gdp"
1428,Andorra,1998,"total_energy_supply, gdp"
1429,Andorra,1999,"total_energy_supply, gdp"


## Validating the cleaned dataset

Checks confirm that the resulting table matches the documented
selection rules and inspection findings.

The checks cover missing values, numerical validity, unique country-year
identifiers, geographic codes, the study period and expected sample size.

Inspection found no duplicate country-year observations and no zero or
negative values among complete observations. These properties are
verified rather than addressed through unnecessary deletion.

Large emissions values are retained because magnitude alone does not
establish an error.

In [6]:
clean_df = (
    clean_df
    .sort_values(["country", "year"])
    .reset_index(drop=True)
)

assert clean_df[identifier_columns].notna().all().all(), (
    "Missing identifiers found."
)

assert clean_df[required_columns].notna().all().all(), (
    "Missing modelling values remain."
)

assert all(
    pd.api.types.is_numeric_dtype(clean_df[column])
    for column in required_columns
), "A modelling variable is not numeric."

assert np.isfinite(
    clean_df[required_columns].to_numpy(dtype=float)
).all(), "Infinite or invalid numerical values found."

assert clean_df[required_columns].gt(0).all().all(), (
    "Unexpected zero or negative values found."
)

assert not clean_df.duplicated(["country", "year"]).any(), (
    "Repeated country-year combinations found."
)

assert clean_df["year"].between(1990, 2022).all(), (
    "A year falls outside the selected period."
)

assert clean_df["iso_code"].astype("string").str.fullmatch(
    r"[A-Z]{3}", na=False
).all(), "A row falls outside the geographic screening rule."

assert rows_before == len(clean_df) + len(excluded_rows)

# Expected counts apply to the inspected dataset version.
assert clean_df.shape == (5358, 7), "Unexpected dataset shape."
assert clean_df["country"].nunique() == 164, "Unexpected entity count."

print("All validation checks passed.")
print("Cleaned shape:", clean_df.shape)

All validation checks passed.
Cleaned shape: (5358, 7)


In [7]:
cleaning_summary = pd.DataFrame(cleaning_log)

cleaning_summary["rows_removed_at_step"] = (
    cleaning_summary["rows"].shift(1)
    - cleaning_summary["rows"]
).fillna(0).astype(int)

display(cleaning_summary)

print("Original DataFrame shape:", df.shape)
print("Cleaned DataFrame shape:", clean_df.shape)

,stage,rows,entities,rows_removed_at_step
0,Original dataset,56269,259,0
1,After geographic screening,48306,223,7963
2,After selecting 1990–2022,7359,223,40947
3,After complete-case selection,5358,164,2001


Original DataFrame shape: (56269, 79)
Cleaned DataFrame shape: (5358, 7)


## Saving the cleaned data and exclusion record

The cleaned dataset is saved separately from the original download.

The missing-data exclusion table and cleaning summary are also saved
to make the selection process traceable.

The cleaned table is ready for evaluation-split design and subsequent
exploration. Scaling and any other learned preprocessing will be fitted
using training data only (scikit-learn developers, n.d.a).

In [8]:
processed_dir = file_path.resolve().parent.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

cleaned_path = processed_dir / "co2_clean_1990_2022.csv"
excluded_path = processed_dir / "co2_excluded_missing_1990_2022.csv"
summary_path = processed_dir / "co2_cleaning_summary.csv"

clean_df.to_csv(cleaned_path, index=False)
excluded_rows.to_csv(excluded_path, index=False)
cleaning_summary.to_csv(summary_path, index=False)

saved_df = pd.read_csv(cleaned_path)

pd.testing.assert_frame_equal(
    clean_df,
    saved_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-12
)

print("Saved and verified:", cleaned_path)
print("Exclusion record:", excluded_path)
print("Cleaning summary:", summary_path)

Saved and verified: C:\Users\franc\Downloads\AIML assessment group\data\processed\co2_clean_1990_2022.csv
Exclusion record: C:\Users\franc\Downloads\AIML assessment group\data\processed\co2_excluded_missing_1990_2022.csv
Cleaning summary: C:\Users\franc\Downloads\AIML assessment group\data\processed\co2_cleaning_summary.csv


In [9]:
print("CLEANING SUMMARY")
display(cleaning_summary)

validation_summary = pd.DataFrame({
    "Check": [
        "Final rows",
        "Final columns",
        "Countries and territories",
        "First year",
        "Last year",
        "Missing required values",
        "Duplicate country-year records",
        "Zero or negative required values",
        "Non-finite required values",
        "Rows excluded for missing values"
    ],
    "Result": [
        len(clean_df),
        clean_df.shape[1],
        clean_df["country"].nunique(),
        clean_df["year"].min(),
        clean_df["year"].max(),
        clean_df[required_columns].isna().sum().sum(),
        clean_df.duplicated(["country", "year"]).sum(),
        clean_df[required_columns].le(0).sum().sum(),
        (~np.isfinite(
            clean_df[required_columns].to_numpy(dtype=float)
        )).sum(),
        len(excluded_rows)
    ],
    "Expected": [
        5358, 7, 164, 1990, 2022, 0, 0, 0, 0, 2001
    ]
})

validation_summary["Passed"] = (
    validation_summary["Result"]
    == validation_summary["Expected"]
)

print("\nVALIDATION SUMMARY")
display(validation_summary)

print(
    "\nAll displayed checks passed:",
    validation_summary["Passed"].all()
)

CLEANING SUMMARY


,stage,rows,entities,rows_removed_at_step
0,Original dataset,56269,259,0
1,After geographic screening,48306,223,7963
2,After selecting 1990–2022,7359,223,40947
3,After complete-case selection,5358,164,2001



VALIDATION SUMMARY


,Check,Result,Expected,Passed
0,Final rows,5358,5358,True
1,Final columns,7,7,True
2,Countries and territories,164,164,True
3,First year,1990,1990,True
4,Last year,2022,2022,True
5,Missing required values,0,0,True
6,Duplicate country-year records,0,0,True
7,Zero or negative required values,0,0,True
8,Non-finite required values,0,0,True
9,Rows excluded for missing values,2001,2001,True



All displayed checks passed: True
